# Train and segment one area

This notebook walks through a complete single-area tutorial:

1. prepare one labeled tile
2. build SAM training chips
3. train a SAM decoder
4. inspect SAM proposal outputs
5. build StarDist training records
6. train StarDist
7. run segmentation

This is a tutorial for understanding and exercising the full pipeline.
It is not a recommended evaluation setup.
For real model assessment, keep separate training and held-out areas.

## Inputs

This notebook assumes the clipped rasters are already in the repo `data/` directory.

You still need to provide a crown-label file for training.
That label file must overlap the clipped rasters.

In [1]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "samstars").exists():
            return candidate
    raise RuntimeError("Could not locate the repo root from the current working directory.")


repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from samstars.data import build_scale_stats_from_proposal_records, build_single_tile
from samstars.segmentation import run_segmentation
from samstars.training.chips import build_training_chips
from samstars.training.records import build_stardist_training_records
from samstars.training.sam_training import train_decoder
from samstars.training.stardist_training import train_stardist_model

/Users/iosefa/miniforge3/envs/samstars/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
bioimageio_utils.py (2): pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.


## Set paths

Update the label and checkpoint paths for your own setup.

In [2]:
data_dir = repo_root / "data"

img = data_dir / "img_clipped.tif"
chm = data_dir / "chm_clipped.tif"
den = data_dir / "den_clipped.tif"
crowns = data_dir / "crowns_training.gpkg"
sam_base_ckpt = repo_root / "sam/sam_vit_l_0b3195.pth"
work_dir = data_dir / "train_and_segment_one_area_run"
device = "cpu"

labeled_tiles_dir = work_dir / "labeled_tiles"
training_chips_dir = work_dir / "training_chips"
training_run_dir = work_dir / "training_run"
records_dir = work_dir / "stardist_training_records"
models_dir = work_dir / "models"
inference_dir = work_dir / "segmentation_run"
sam_ckpt = models_dir / "sam_crowns_decoder.pth"
stardist_model = models_dir / "stardist"

print("repo_root:", repo_root)
print("data_dir:", data_dir)

repo_root: /Users/iosefa/repos/sam
data_dir: /Users/iosefa/repos/sam/data


## Set tutorial parameters

These are explicit on purpose. Do not treat them as fixed project defaults. Change them, rerun the notebook, and compare the intermediate proposal outputs before committing to StarDist training.

`rgb_bands` controls which three image bands are written into the prepared tile. For a 3-band raster, keep `(1, 2, 3)`. For a multiband raster, choose the three bands you want the full workflow to use.


In [3]:
rgb_bands = (1, 2, 3)

chip_size = 512
chip_stride = 384 # 256

sam_epochs = 15
stardist_epochs = 50
val_fraction = 0.2

crop_smooth_sigma = 2.0
smooth_sigma = 4.0
post_smooth_sigma = 0.0

stardist_input_key = "index"
use_aux = True


## Prepare one labeled tile

This creates the tile layout used by the downstream training helpers.

In [4]:
build_single_tile(
    img=img,
    chm=chm,
    den=den,
    crowns=crowns,
    rgb_bands=rgb_bands,
    out=labeled_tiles_dir,
    skip_existing=True,
)


function_base.py (4824): Warning: 'partition' will ignore the 'mask' of the MaskedArray.
cost_surface.py (137): No SLIC provided – cost built from 3 terms only.
_base.py (248): Precomputed sparse input was not sorted by row values. Use the function sklearn.neighbors.sort_graph_by_row_values to sort the input by row values, with warn_when_not_sorted=False to remove this warning.


✓ Single-tile inputs written to /Users/iosefa/repos/sam/data/train_and_segment_one_area_run/labeled_tiles


PosixPath('/Users/iosefa/repos/sam/data/train_and_segment_one_area_run/labeled_tiles')

## Build SAM training chips

In [5]:
build_training_chips(
    tiles_dir=labeled_tiles_dir,
    out_dir=training_chips_dir,
    chip_size=chip_size,
    stride=chip_stride,
)

tile_000_000: wrote 180 training chips
Built 180 training chips in /Users/iosefa/repos/sam/data/train_and_segment_one_area_run/training_chips


180

## Train models

This section first trains the SAM decoder, then creates and inspects proposal rasters, and only then trains StarDist.

In [6]:
train_decoder(
    chip_dir=training_chips_dir,
    ckpt=sam_base_ckpt,
    out_path=sam_ckpt,
    epochs=sam_epochs,
    device=device,
)

training_chips: epochs=15 (iters/epoch=180)
training_chips epoch 1/15 loss=0.5246
training_chips epoch 2/15 loss=0.3297
training_chips epoch 3/15 loss=0.2563
training_chips epoch 4/15 loss=0.2383
training_chips epoch 5/15 loss=0.2113
training_chips epoch 6/15 loss=0.1955
training_chips epoch 7/15 loss=0.1970
training_chips epoch 8/15 loss=0.1869
training_chips epoch 9/15 loss=0.1739
training_chips epoch 10/15 loss=0.1598
training_chips epoch 11/15 loss=0.1598
training_chips epoch 12/15 loss=0.1544
training_chips epoch 13/15 loss=0.1554
training_chips epoch 14/15 loss=0.1650
training_chips epoch 15/15 loss=0.1539
✓ weights → /Users/iosefa/repos/sam/data/train_and_segment_one_area_run/models/sam_crowns_decoder.pth


In [ ]:
proposal_result = run_segmentation(
    tiles_dir=labeled_tiles_dir,
    out_dir=training_run_dir,
    sam_ckpt=sam_ckpt,
    device=device,
    run_sam=True,
    run_stardist=False,
    crop_smooth_sigma=crop_smooth_sigma,
    smooth_sigma=smooth_sigma,
    post_smooth_sigma=post_smooth_sigma,
    fast=False
)

seeds in chip: 5395
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.06074980e-06 9.23150987e-01
 9.99992788e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.06074980e-06 9.27315354e-01
 9.99992788e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.06074980e-06 7.23024485e-01
 9.99991834e-01]
seed prob pcts: [1.58057983e-10 2.02361861e-07 3.18549087e-06 8.65363877e-01
 9.95469332e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.06074980e-06 8.91041622e-01
 9.99992788e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.06074980e-06 8.94892039e-01
 9.98421609e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.06074980e-06 9.15126873e-01
 9.99992788e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.06074980e-06 9.23051921e-01
 9.99992788e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.17279421e-06 8.95088379e-01
 9.99992788e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.06074980e-06 9.23208594e-01
 9.99992788e-01]
seed prob pcts: [1.16361983e-10 1.893278

## Inspect SAM proposal outputs

Inspect the proposal rasters before continuing.
If the proposals are poor, adjust the parameters above and rerun this section before training StarDist.

In [ ]:
proposal_result

In [ ]:
proposal_rasters_dir = proposal_result["proposals_root"] / "rasters"
sorted(p.name for p in proposal_rasters_dir.glob("*.tif"))[:20]

## Build StarDist training records

In [ ]:
train_records, val_records = build_stardist_training_records(
    proposal_records=proposal_result["proposal_records"],
    tiles_dir=labeled_tiles_dir,
    out_dir=records_dir,
    val_fraction=val_fraction,
)

print("train records:", len(__import__('json').load(open(train_records))))
print("val records:", len(__import__('json').load(open(val_records))))

In [ ]:
scale_stats = work_dir / "training_scale_stats.json"

build_scale_stats_from_proposal_records(
    proposal_records=proposal_result["proposal_records"],
    out=scale_stats,
    percentiles=(2.0, 98.0),
    chm_key="chm",
    nodata_values=(-9999.0,),
)

In [ ]:
train_stardist_model(
    train_records=train_records,
    val_records=val_records,
    out_dir=stardist_model.parent,
    epochs=stardist_epochs,
    input_key=stardist_input_key,
    use_aux=use_aux,
    scale_stats_path=scale_stats,
)

## Run segmentation

In [ ]:
result = run_segmentation(
    img=img,
    chm=chm,
    den=den,
    out_dir=inference_dir,
    sam_ckpt=sam_ckpt,
    stardist_model=stardist_model,
    stardist_input_key=stardist_input_key,
    device=device,
    crop_smooth_sigma=crop_smooth_sigma,
    smooth_sigma=smooth_sigma,
    post_smooth_sigma=post_smooth_sigma,
)

## Final output

In [ ]:
inference_dir / "crowns" / "crowns.gpkg"